# Belief-PDDL Getting Started: Blocksworld Toy Domain

In this notebook, we demonstrate the core mathematical breakthrough of the **Belief-PDDL** research scaffold: How to track continuous probabilities inside a discrete symbolic world, and how to use Operations Research (CP-SAT) to purge physics hallucinations.

While the full framework uses OpenAI CLIP and Unity (ALFWorld), this notebook cleanly maps the concept in pure math using the `Blocksworld` domain so you can review the constraint algorithms without requiring a GUI server.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.belief.state import PredicateBelief
from src.belief.update import BeliefUpdater
from src.belief.projection import BeliefProjector

print("Neuro-Symbolic Core Loaded!")

## 1. Setting up Log-Odds Tracking
We start by initializing a Belief matrix and assigning probabilities to our blocks. For this test, let's pretend our neural Vision Language Model thinks `block_A` is **both in our hand (holding) AND sitting on the table (on-table)** at the exact same time.

This is a classic VLM hallucination. Standard Pyperplan logic will crash here.

In [ ]:
belief_state = PredicateBelief()
updater = BeliefUpdater(alpha=1.0)

# Simulating the VLM soft-logits:
raw_probs = {
    "holding(agent,block_A)": 0.95, # 95% sure it's in our hand
    "on-table(block_A)": 0.80,     # 80% sure it's still on the table 
    "clear(block_A)": 0.98         # 98% sure nothing is on top of it
}

# Update log odds matrices
for pred, prob in raw_probs.items():
    n = updater.update(belief_state.get_belief(pred), obs_prob=prob)
    belief_state.set_belief(pred, n, timestep=1)
    
print("Corrupted Neural Probabilities:", belief_state.probs)

## 2. Google OR-Tools (CP-SAT) Projection
Now, we pass this corrupted matrix into the mathematical firewall. 
Our `constraints.yaml` explicitly dictates: `mutual_exclusion(["holding(agent,X)", "on-table(X)"])`.

The generic CP-SAT Boolean Constraints Optimizer will calculate the *Maximum A Posteriori (MAP)* universe summing the highest possible mathematically valid state based on the probabilities.

In [ ]:
projector = BeliefProjector("../domains/blocksworld/constraints.yaml")

projected_universe = projector.project_map_state(belief_state.probs)

print("\n--- Mathematically Purged MAP State ---")
for pred, is_true in projected_universe.items():
    if is_true:
        print(f"✅ {pred} is TRUE")
    else:
        print(f"❌ {pred} is FALSE")

print("\nNotice how the `on-table` parameter was successfully severed because `holding(0.95) > on-table(0.80)`!")
print("This is the flawless logical state injected into Pyperplan.")